# Kyrgyzstan Education Statistics — Forecasting 2025–2030

**Methods:** Linear regression, Polynomial regression (degree 2), ARIMA  
**Targets:** International student enrollment (CIS, Non-CIS, Total) · Education budget  
**Horizon:** 2025–2030

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_absolute_error
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

BLUE   = '#2563EB'
ORANGE = '#EA580C'
GREEN  = '#16A34A'
RED    = '#DC2626'
GRAY   = '#6B7280'

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})
print('Setup complete.')

## 1. Load & Prepare Data

In [ ]:
cis_raw     = pd.read_csv('students_cis.csv',     encoding='utf-8-sig')
non_cis_raw = pd.read_csv('students_non_cis.csv', encoding='utf-8-sig')
budget_raw  = pd.read_csv('education_budget.csv', encoding='utf-8-sig')

year_cols = [str(y) for y in range(2010, 2025)]

def extract_totals(df):
    en_col = df.columns[2]
    mask = df[en_col].astype(str).str.strip().str.lower().isin(['total'])
    row = df[mask].iloc[0]
    return pd.to_numeric(row[year_cols], errors='coerce')

cis_total     = extract_totals(cis_raw)
non_cis_total = extract_totals(non_cis_raw)

students_df = pd.DataFrame({
    'Year':    [int(y) for y in year_cols],
    'CIS':     cis_total.values,
    'Non_CIS': non_cis_total.values
})
students_df['Total'] = students_df['CIS'] + students_df['Non_CIS']

# Budget
budget = budget_raw.copy()
en_col_b = budget.columns[2]
budget[en_col_b] = budget[en_col_b].astype(str).str.strip()
b_years = [str(y) for y in range(2001, 2025)]
for col in b_years:
    if col in budget.columns:
        budget[col] = pd.to_numeric(budget[col], errors='coerce')

def get_budget_row(kw):
    mask = budget[en_col_b].str.lower().str.contains(kw, na=False)
    return budget[mask].iloc[0][b_years].astype(float) if mask.any() else pd.Series([np.nan]*len(b_years), index=b_years)

budget_ts = pd.DataFrame({
    'Year': [int(y) for y in b_years],
    'Total_mln_som': get_budget_row('total').values,
})

print('Students data (2010-2024):')
display(students_df)
print('\nBudget rows:', len(budget_ts))

## 2. Forecasting Helpers

In [ ]:
FUTURE_YEARS = list(range(2025, 2031))

def fit_and_forecast(years, values, future_years, degree=2):
    """Fit linear + polynomial regression and return predictions with metrics."""
    X  = np.array(years).reshape(-1, 1)
    y  = np.array(values)
    Xf = np.array(future_years).reshape(-1, 1)

    # Linear
    lin_reg = LinearRegression().fit(X, y)
    lin_pred = lin_reg.predict(Xf)
    lin_r2   = r2_score(y, lin_reg.predict(X))
    lin_mae  = mean_absolute_error(y, lin_reg.predict(X))

    # Polynomial
    poly_feat = PolynomialFeatures(degree)
    Xp  = poly_feat.fit_transform(X)
    Xfp = poly_feat.transform(Xf)
    poly_reg  = LinearRegression().fit(Xp, y)
    poly_pred = poly_reg.predict(Xfp)
    poly_r2   = r2_score(y, poly_reg.predict(Xp))
    poly_mae  = mean_absolute_error(y, poly_reg.predict(Xp))

    return {
        'linear':  {'pred': lin_pred,  'r2': lin_r2,  'mae': lin_mae},
        'poly':    {'pred': poly_pred, 'r2': poly_r2, 'mae': poly_mae},
    }


def fit_arima(series, future_years, order=(1,1,1)):
    """Fit ARIMA and return forecast for future_years."""
    model = ARIMA(series, order=order).fit()
    fc    = model.forecast(steps=len(future_years))
    aic   = model.aic
    return {'pred': fc.values, 'aic': aic, 'model': model}

print('Helper functions defined.')

## 3. Total International Students — Forecast

In [ ]:
total_fc  = fit_and_forecast(students_df['Year'], students_df['Total'], FUTURE_YEARS)
arima_fc  = fit_arima(students_df['Total'].values, FUTURE_YEARS, order=(1,1,0))

print(f"Linear regression:      R² = {total_fc['linear']['r2']:.4f}  |  MAE = {total_fc['linear']['mae']:,.0f}")
print(f"Polynomial (degree 2):  R² = {total_fc['poly']['r2']:.4f}  |  MAE = {total_fc['poly']['mae']:,.0f}")
print(f"ARIMA(1,1,0):           AIC = {arima_fc['aic']:.2f}")

forecast_total = pd.DataFrame({
    'Year':          FUTURE_YEARS,
    'Linear':        total_fc['linear']['pred'].round(0).astype(int),
    'Poly_deg2':     total_fc['poly']['pred'].round(0).astype(int),
    'ARIMA(1,1,0)':  arima_fc['pred'].round(0).astype(int),
})
display(forecast_total)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

# Historical
ax.plot(students_df['Year'], students_df['Total'], 'o-', color=GRAY, linewidth=2.5,
        markersize=6, label='Historical (2010–2024)', zorder=5)

# Forecasts
ax.plot(FUTURE_YEARS, total_fc['linear']['pred'], 's--', color=BLUE,
        linewidth=2, markersize=7, label=f"Linear  (R²={total_fc['linear']['r2']:.3f})")
ax.plot(FUTURE_YEARS, total_fc['poly']['pred'], '^--', color=ORANGE,
        linewidth=2, markersize=7, label=f"Poly-2  (R²={total_fc['poly']['r2']:.3f})")
ax.plot(FUTURE_YEARS, arima_fc['pred'], 'D--', color=GREEN,
        linewidth=2, markersize=7, label=f"ARIMA(1,1,0)  AIC={arima_fc['aic']:.0f}")

ax.axvline(2024.5, color='black', linestyle=':', linewidth=1, alpha=0.5)
ax.text(2024.6, ax.get_ylim()[0]*1.05, 'Forecast →', fontsize=9, color='black', alpha=0.6)

ax.set_title('Total International Students — Kyrgyzstan Forecast 2025–2030',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Students')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend(framealpha=0.3)
plt.tight_layout()
plt.savefig('forecast_total_students.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 4. CIS vs Non-CIS Separate Forecasts

In [ ]:
cis_fc     = fit_and_forecast(students_df['Year'], students_df['CIS'],     FUTURE_YEARS)
non_cis_fc = fit_and_forecast(students_df['Year'], students_df['Non_CIS'], FUTURE_YEARS)

arima_cis     = fit_arima(students_df['CIS'].values,     FUTURE_YEARS, order=(1,1,0))
arima_non_cis = fit_arima(students_df['Non_CIS'].values, FUTURE_YEARS, order=(1,1,0))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, df_col, fc, arima_r, title, col in [
    (axes[0], 'CIS',     cis_fc,     arima_cis,     'CIS Countries',     BLUE),
    (axes[1], 'Non_CIS', non_cis_fc, arima_non_cis, 'Non-CIS Countries', ORANGE),
]:
    ax.plot(students_df['Year'], students_df[df_col], 'o-', color=GRAY,
            linewidth=2.5, markersize=6, label='Historical', zorder=5)
    ax.plot(FUTURE_YEARS, fc['linear']['pred'], 's--', color=col,
            linewidth=2, markersize=7, label=f"Linear (R²={fc['linear']['r2']:.3f})")
    ax.plot(FUTURE_YEARS, fc['poly']['pred'], '^--', color=RED,
            linewidth=2, markersize=7, label=f"Poly-2 (R²={fc['poly']['r2']:.3f})")
    ax.plot(FUTURE_YEARS, arima_r['pred'], 'D--', color=GREEN,
            linewidth=2, markersize=7, label=f"ARIMA")
    ax.axvline(2024.5, color='black', linestyle=':', linewidth=1, alpha=0.4)
    ax.set_title(f'{title} — Forecast 2025–2030', fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Students')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
    ax.legend(framealpha=0.3, fontsize=9)

plt.tight_layout()
plt.savefig('forecast_cis_noncis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 5. Education Budget Forecast

In [ ]:
budget_clean = budget_ts.dropna(subset=['Total_mln_som'])
budget_fc    = fit_and_forecast(budget_clean['Year'], budget_clean['Total_mln_som'], FUTURE_YEARS)
arima_budget = fit_arima(budget_clean['Total_mln_som'].values, FUTURE_YEARS, order=(1,1,1))

print(f"Budget Linear:   R² = {budget_fc['linear']['r2']:.4f}  |  MAE = {budget_fc['linear']['mae']:,.1f} mln som")
print(f"Budget Poly-2:   R² = {budget_fc['poly']['r2']:.4f}  |  MAE = {budget_fc['poly']['mae']:,.1f} mln som")
print(f"Budget ARIMA:    AIC = {arima_budget['aic']:.2f}")

forecast_budget = pd.DataFrame({
    'Year':              FUTURE_YEARS,
    'Linear_mln_som':    budget_fc['linear']['pred'].round(1),
    'Poly2_mln_som':     budget_fc['poly']['pred'].round(1),
    'ARIMA_mln_som':     arima_budget['pred'].round(1),
})
display(forecast_budget)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ax.fill_between(budget_clean['Year'], budget_clean['Total_mln_som'],
                alpha=0.15, color=BLUE)
ax.plot(budget_clean['Year'], budget_clean['Total_mln_som'], 'o-', color=GRAY,
        linewidth=2.5, markersize=6, label='Historical (2001–2024)', zorder=5)
ax.plot(FUTURE_YEARS, budget_fc['linear']['pred'], 's--', color=BLUE,
        linewidth=2, markersize=7, label=f"Linear  (R²={budget_fc['linear']['r2']:.3f})")
ax.plot(FUTURE_YEARS, budget_fc['poly']['pred'], '^--', color=ORANGE,
        linewidth=2, markersize=7, label=f"Poly-2  (R²={budget_fc['poly']['r2']:.3f})")
ax.plot(FUTURE_YEARS, arima_budget['pred'], 'D--', color=GREEN,
        linewidth=2, markersize=7, label=f"ARIMA(1,1,1)")

ax.axvline(2024.5, color='black', linestyle=':', linewidth=1, alpha=0.5)
ax.set_title('Education Budget — Kyrgyzstan Forecast 2025–2030 (mln som)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('mln som')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax.legend(framealpha=0.3)
plt.tight_layout()
plt.savefig('forecast_budget.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 6. Combined Forecast Table (Export)

In [ ]:
all_forecasts = pd.DataFrame({
    'Year':                     FUTURE_YEARS,
    'CIS_Linear':               cis_fc['linear']['pred'].round(0).astype(int),
    'CIS_Poly2':                cis_fc['poly']['pred'].round(0).astype(int),
    'CIS_ARIMA':                arima_cis['pred'].round(0).astype(int),
    'NonCIS_Linear':            non_cis_fc['linear']['pred'].round(0).astype(int),
    'NonCIS_Poly2':             non_cis_fc['poly']['pred'].round(0).astype(int),
    'NonCIS_ARIMA':             arima_non_cis['pred'].round(0).astype(int),
    'Total_Linear':             total_fc['linear']['pred'].round(0).astype(int),
    'Total_Poly2':              total_fc['poly']['pred'].round(0).astype(int),
    'Total_ARIMA':              arima_fc['pred'].round(0).astype(int),
    'Budget_Linear_mln_som':    budget_fc['linear']['pred'].round(1),
    'Budget_Poly2_mln_som':     budget_fc['poly']['pred'].round(1),
    'Budget_ARIMA_mln_som':     arima_budget['pred'].round(1),
})
all_forecasts.to_csv('forecasts_2025_2030.csv', index=False)
display(all_forecasts)
print('\nForecasts exported to forecasts_2025_2030.csv')

## 7. Interpretation

- **Linear regression** assumes the trend of the full 2010–2024 period continues unchanged. Given the sharp COVID spike (2020–2021), it overestimates future growth.
- **Polynomial (degree 2)** captures the parabolic rise-and-fall pattern more faithfully, but extrapolates aggressively beyond the data range.
- **ARIMA(1,1,0)** treats the series as a random walk with drift, fitting the most recent momentum. It tends to be the most conservative and is generally preferred for short horizons.
- **Recommendation:** Use ARIMA projections as the base scenario; use Linear as a lower bound and Poly-2 as an upper bound.